# 7주차 — DDL — 테이블 설계와 생성
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다.
어떤 셀이 오류로 막히거나 데이터가 엉켰다면, 아무 셀에서 `reset_db()`를 실행하면 처음 상태로 돌아갑니다.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, shutil, sqlite3, urllib.request
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/data-v1/mini_emr/mini_emr.sqlite"
DB_PATH = "mini_emr.sqlite"
WORK_PATH = "mini_emr_practice.sqlite"   # 실습에 사용할 파일

con = None

def reset_db():
    """DB를 원본 상태로 되돌리고 다시 연결 (막히면 언제든 호출)"""
    global con
    if con is not None:
        con.close()
    try:
        urllib.request.urlretrieve(DB_URL, DB_PATH)
    except Exception as e:
        if not os.path.exists(DB_PATH):
            raise RuntimeError(
                "DB 다운로드 실패 — 왼쪽 파일 탭에 mini_emr.sqlite를 업로드한 뒤 이 셀을 다시 실행하세요."
            ) from e
        print("다운로드 실패 — 업로드되어 있는 파일을 사용합니다.")
    if WORK_PATH != DB_PATH:
        shutil.copyfile(DB_PATH, WORK_PATH)   # 원본 보존, 실습은 사본에서
    con = sqlite3.connect(WORK_PATH)
    con.execute("PRAGMA foreign_keys = ON")   # SQLite는 기본으로 꺼져 있음 — 반드시 켠다
    assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1

def q(sql):
    """조회(SELECT) 실행 → 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """데이터/구조를 바꾸는 SQL(INSERT/UPDATE/DELETE/CREATE 등) 실행.
    여러 문장을 순서대로 실행하고 즉시 커밋합니다 — 트랜잭션(BEGIN/ROLLBACK) 실습은
    8주차처럼 con.cursor()를 직접 사용하세요."""
    con.executescript(sql); con.commit(); print("실행 완료")

reset_db()
q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. CREATE TABLE — 제약조건과 함께
이 노트북은 **사본 DB**(`mini_emr_practice.sqlite`)에서 실습합니다 — 자유롭게 만들었다 지워도 됩니다. 막히면 `reset_db()`.

In [ ]:
# 코호트 테이블 생성
run("""
CREATE TABLE IF NOT EXISTS my_cohort (
  subject_id   TEXT PRIMARY KEY,
  patient_id   TEXT NOT NULL REFERENCES patient(patient_id),
  index_date   DATE NOT NULL,
  age_at_index INTEGER CHECK (age_at_index BETWEEN 0 AND 120),
  sex          TEXT NOT NULL CHECK (sex IN ('M','F')),
  dm_flag      INTEGER NOT NULL DEFAULT 0 CHECK (dm_flag IN (0,1))
);
""")

In [ ]:
# 생성 확인
q("""
SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1
""")

## 2. 제약조건이 일하는 순간 — 아래 셀은 **일부러 에러**가 납니다

In [ ]:
# CHECK 위반 실험: SpO2 250%는 입력 자체가 거부된다
try:
    run("""
    CREATE TABLE IF NOT EXISTS vital_demo (visit_id TEXT, spo2 INTEGER CHECK (spo2 BETWEEN 0 AND 100));
    INSERT INTO vital_demo VALUES ('V0001', 250);
    """)
except Exception as e:
    print('거부됨 →', e)

## 3. ALTER / DROP

In [ ]:
# 컬럼 추가 후 테이블 정리 — 이 셀을 두 번 실행하면 duplicate column 에러 (이미 있는 컬럼은 재추가 불가 — reset_db() 후 재실행)
run("""
ALTER TABLE my_cohort ADD COLUMN htn_flag INTEGER DEFAULT 0;
DROP TABLE IF EXISTS vital_demo;
""")

## 연습문제
DDL 문제는 결과 표가 없으므로 `q()`가 아닌 `run()` 안에 작성합니다.

In [ ]:
# [연습 Q1] '수술(operation)' 테이블을 설계해 CREATE 하라 — 수술ID(PK), 내원ID(FK), 수술명(NOT NULL), 수술일, 집도의ID(FK)
run("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] operation에 마취종류 컬럼을 ALTER로 추가해 보라 — CHECK 제약까지 걸려면 왜 어려운지 겪어보기
run("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] 자기 연구 주제의 코호트 테이블을 컬럼 6개 이상으로 설계하라 (과제 제출용)
run("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. 수술 테이블**
```sql
CREATE TABLE operation (
  op_id TEXT PRIMARY KEY,
  visit_id TEXT NOT NULL REFERENCES visit(visit_id),
  op_name TEXT NOT NULL,
  op_date DATE,
  surgeon_id TEXT REFERENCES staff(staff_id)
);
```

**A2. 마취종류 추가**
```sql
-- SQLite의 ALTER는 CHECK 추가를 지원하지 않음 → 테이블 재생성 필요.
-- 교훈: 제약조건은 처음 설계할 때 넣는 것이 훨씬 싸다.
ALTER TABLE operation ADD COLUMN anesthesia TEXT;
```